In [34]:
from crossmatching import FileCatalog, Crossmatcher, SimbadIdSupplier, EMCCatalog, EMCIdSupplier
from crossmatching.enrichment import ParamFiller, NeaParamSource
import astropy.units as u
from astropy.table import Table

fc = FileCatalog(
    "./input/HPIC_LC4_combined_d50.txt",
    ra_key="ra",
    ra_unit=u.deg,
    dec_key="dec",
    dec_unit=u.deg,
    planet_uid="star_name",
    host_key="star_name"
)

cmf = Crossmatcher(fc, SimbadIdSupplier())

cmf.load_catalog()


t = Table()
t["name"] = [
    "HR 8799",
    "Beta Pictoris",
    "51 Eridani",
    "HIP 65426",
    "PDS 70",
    "HR 2562",
    "HD 95086",
    "HIP 99770",
    "AF Lep",
]

result = cmf.id_crossmatch(t, input_starname_key="name")


In [42]:
d = result["name", "star_name", "sy_dist", "st_spectype"].copy()
d.rename_column("star_name", "star_name (hpic)")
d["sy_dist"] = d["sy_dist"].round(1)
d.pprint_all()

     name     star_name (hpic) sy_dist st_spectype
------------- ---------------- ------- -----------
      HR 8799    TIC 245368902    40.8  F0+VkA5mA5
Beta Pictoris    TIC 270577175    19.6         A6V
   51 Eridani    TIC 298810162    29.9        F0IV
      HR 2562    TIC 167656187    33.9   F5VFe+0.4
    HIP 99770     TIC 10988057    40.7         A2V
       AF Lep     TIC 94945758    26.8    F8V(n)k:


In [ ]:
missing = list(set(t["name"]) - set(result["name"]))
print(missing)

t2 = Table()
t2["name"] = missing

cme = Crossmatcher(EMCCatalog(), EMCIdSupplier())
cme.load_catalog(from_file="./exo-mercat.csv")
cme.load_alternate_ids(missing, from_file="./exo-mercat.csv")
re2 = cme.id_crossmatch(t2, input_starname_key="name")


[np.str_('HIP 65426'), np.str_('PDS 70'), np.str_('HD 95086')]


In [ ]:
nea_src = NeaParamSource()
nea_src.load(from_file='./pscomppars.txt', format='ascii')
merger   = ParamFiller([nea_src])

e_emc = merger.enrich(
    re2,
    input_starname_key="star_name", 
    id_supplier=cme.id_supplier, 
    alternate_ids=cme.alternate_ids,
    **EMCCatalog.ENRICH_KEYS
)

d2 = e_emc["name", "st_spectype", "sy_dist"].copy()
d2.rename_column("st_spectype")


   name   st_spectype sy_dist
--------- ----------- -------
HIP 65426        A2 V 108.875
   PDS 70          K7 113.064
   PDS 70          K7 113.064
   PDS 70                  --
 HD 95086      A8 III 86.2279


In [44]:
cmf.coordinate_crossmatch(re2["name", "main_id_ra", "main_id_dec"], input_starname_key="name", ra_key="main_id_ra", dec_key="main_id_dec")

name,main_id_ra,main_id_dec,star_name,sy_dist,st_spectype,st_rad,st_teff,st_mass,st_age,ra,dec,sy_vmag,sy_jmag,sy_hmag,sy_kmag,known_binary_fl,gaia_binary_fl,WDSsep,wds_deltamag,match_type,crossmatching_angular_separation
,,,,,,,,,,,,,,,,,,,,,arcsec
str9,float64,float64,str29,float64,str18,float64,str18,float64,str18,float64,float64,str18,str18,str18,str18,str5,str4,str18,str19,str11,float64
